# 梯度下降法中的 Adam 优化器详解

本 Notebook 系统梳理 Adam 优化算法的原理、核心机制、优势与局限，并通过精心设计的目标函数展示其在不同地形下的表现。

## 0. 基本档案

| 项目 | 内容 |
| :--- | :--- |
| **全称** | Adaptive Moment Estimation |
| **中文译名** | 自适应矩估计 |
| **提出者** | Diederik P. Kingma & Jimmy Ba |
| **提出年份** | 2014年 |
| **发表形式** | 会议论文（ICLR 2015） |
| **所属家族** | 自适应学习率算法 + 动量法（融合一阶矩与二阶矩估计） |
| **核心创新** | 同时维护梯度的一阶矩（动量）和二阶矩（自适应学习率），并引入偏差校正，使训练初期更稳定；结合了 Momentum 和 RMSProp 两者的优势 |
| **理论收敛性** | 在凸和非凸条件下均具有理论收敛保证；在适度稀疏梯度和噪声环境下收敛性能稳定，实际应用中收敛速度快 |
| **主要局限** | 部分场景下泛化性能可能不如带冲量的 SGD；对超参数（尤其是学习率）仍有一定敏感性；后期可能因自适应学习率累积导致更新步长过小；未完全解决部分非凸问题中的收敛震荡问题 |

## 1. Adam 是什么？

**Adam（Adaptive Moment Estimation）** 是目前深度学习中 **最主流、最常用的优化算法之一**。

你可以把它理解成一个聪明的“下山”策略，它结合了两种优秀算法的思想：**动量**（Momentum）和 **自适应学习率**（RMSProp）。


---

### 算例一：高度病态凸函数——当 RMSProp 陷入 Zigzag 而 Adam 势如破竹

上面的多地形算例展示了 Adam 跨越鞍点和局部极小的能力，但它还有另一个更为著名的优势：**在梯度条件数极大（即某方向极陡、另一方向极平）的病态地形上，Adam 能有效抑制震荡并持续前进**。

为此，我们构造一个**高度病态凸二次函数**，其 Hessian 矩阵的两个特征值分别为 \(\lambda_x=1\) 和 \(\lambda_y=100\)（相差两个数量级）。这意味着 Y 轴方向极其陡峭，而 X 轴方向非常平缓——这正是 RMSProp 容易产生锯齿形（Zigzag）震荡并卡死的典型场景。

下面，我们让 **Adam** 与 **RMSProp** 从同一起点出发，在完全相同的学习率下进行对比。

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 【全局超参配置区】 - 高度病态凸二次函数测试场
# ============================================================
CONFIG = {
    "start_x": -2.0,           
    "start_y": 2.0,            # 起点放在陡峭的y轴方向，最容易看到Zigzag
    "steps": 2000,              
    
    # 绝对公平参数
    "lr": 0.1,                 
    "beta1": 0.9,
    "beta2": 0.999,
    
    "x_min": -4.0,
    "x_max": 4.0,
    "y_min": -4.0,
    "y_max": 4.0,
    "mesh_res": 400,
    
    "fig_height": 700,
    "fig_width": 1000,
    "path_line_width": 2.5,
    "path_marker_size": 4,     
    "title_x": 0.02,
}

# 高度病态凸系数（特征值极度悬殊）
LAMBDA_X = 1.0   # 平缓方向（X轴）
LAMBDA_Y = 100.0     # 陡峭方向（Y轴）

def f(x, y):
    return 0.5 * (LAMBDA_X * x**2 + LAMBDA_Y * y**2)

def grad_f(x, y):
    dx = LAMBDA_X * x
    dy = LAMBDA_Y * y
    return dx, dy

# ============================================================
# RMSProp 优化器
# ============================================================
def rmsprop_optimizer(start_x, start_y, steps):
    lr = CONFIG["lr"]
    x, y = start_x, start_y
    sx, sy = 0.0, 0.0
    path = [(x, y, f(x, y))]
    for _ in range(steps):
        gx, gy = grad_f(x, y)
        sx = 0.9 * sx + (1 - 0.9) * gx**2
        sy = 0.9 * sy + (1 - 0.9) * gy**2
        x = x - lr * gx / (np.sqrt(sx) + 1e-8)
        y = y - lr * gy / (np.sqrt(sy) + 1e-8)
        path.append((x, y, f(x, y)))
    return np.array(path)

# ============================================================
# Adam 优化器
# ============================================================
def adam_optimizer(start_x, start_y, steps):
    lr = CONFIG["lr"]
    x, y = start_x, start_y
    mx, my = 0.0, 0.0
    vx, vy = 0.0, 0.0
    t = 0
    path = [(x, y, f(x, y))]
    for _ in range(steps):
        t += 1
        gx, gy = grad_f(x, y)
        mx = 0.9 * mx + (1 - 0.9) * gx
        my = 0.9 * my + (1 - 0.9) * gy
        vx = 0.999 * vx + (1 - 0.999) * gx**2
        vy = 0.999 * vy + (1 - 0.999) * gy**2
        m_hat_x = mx / (1 - 0.9**t)
        m_hat_y = my / (1 - 0.9**t)
        v_hat_x = vx / (1 - 0.999**t)
        v_hat_y = vy / (1 - 0.999**t)
        x = x - lr * m_hat_x / (np.sqrt(v_hat_x) + 1e-8)
        y = y - lr * m_hat_y / (np.sqrt(v_hat_y) + 1e-8)
        path.append((x, y, f(x, y)))
    return np.array(path)

start_x = CONFIG["start_x"]
start_y = CONFIG["start_y"]
common_steps = CONFIG["steps"]

print("=" * 70)
print("【高度病态凸理论实验】")
print("=" * 70)
print(f"函数特征值: λ_x = {LAMBDA_X}, λ_y = {LAMBDA_Y} (极度病态)")
print(f"绝对公平学习率: α = {CONFIG['lr']}")
print(f"迭代步数: {common_steps}")
print("=" * 70)

path_rms = rmsprop_optimizer(start_x, start_y, common_steps)
path_adam = adam_optimizer(start_x, start_y, common_steps)

# ============================================================
# 生成等高线数据（拉长椭圆）
# ============================================================
xs = np.linspace(CONFIG["x_min"], CONFIG["x_max"], CONFIG["mesh_res"])
ys = np.linspace(CONFIG["y_min"], CONFIG["y_max"], CONFIG["mesh_res"])
X, Y = np.meshgrid(xs, ys)
Z = f(X, Y)

# ============================================================
# 绘制图1：等高线路径对比
# ============================================================
fig1 = go.Figure()

fig1.add_trace(go.Contour(
    x=xs, y=ys, z=Z,
    colorscale="Viridis",
    contours=dict(start=0, end=300, size=10, showlabels=False, coloring='fill'),
    line=dict(width=1.2, color='black', dash='solid'),
    showscale=True, opacity=0.7, name="等高线（高度病态凸）"
))

# RMSProp：青色
fig1.add_trace(go.Scatter(x=path_rms[:, 0], y=path_rms[:, 1], mode="lines+markers",
                          line=dict(color="cyan", width=CONFIG["path_line_width"], dash="solid"),
                          marker=dict(size=CONFIG["path_marker_size"], color="cyan", symbol="circle"), 
                          name="RMSProp"))

# Adam：橙色
fig1.add_trace(go.Scatter(x=path_adam[:, 0], y=path_adam[:, 1], mode="lines+markers",
                          line=dict(color="orange", width=CONFIG["path_line_width"], dash="dash"),
                          marker=dict(size=CONFIG["path_marker_size"], color="orange", symbol="square"), 
                          name="Adam"))

fig1.add_trace(go.Scatter(x=[start_x], y=[start_y], mode="markers", marker=dict(size=12, color="#ffcc00"), name="起点"))
fig1.add_trace(go.Scatter(x=[0], y=[0], mode="markers", marker=dict(size=14, color="#00ff00", symbol="star", line=dict(color="black", width=1)), name="全局最优"))

fig1.update_layout(
    width=CONFIG["fig_width"], height=CONFIG["fig_height"], template="plotly_white",
    title=dict(text="迭代路径", x=CONFIG["title_x"], xanchor="left", font=dict(size=18)),
    xaxis_title="x", yaxis_title="y",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5, font=dict(size=11)), # 去掉了 bgcolor 和 bordercolor
    hovermode='closest', margin=dict(t=80, r=80)
)
fig1.show()

# ============================================================
# 绘制图2：收敛曲线
# ============================================================
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=list(range(len(path_rms))), y=path_rms[:, 2], mode='lines', name='RMSProp',
    line=dict(color='cyan', width=2.5, dash='solid'),
    hovertemplate='RMSProp<br>步数: %{x}<br>损失: %{y:.4f}<extra></extra>'
))

fig2.add_trace(go.Scatter(
    x=list(range(len(path_adam))), y=path_adam[:, 2], mode='lines', name='Adam',
    line=dict(color='orange', width=2.5, dash='dash'),
    hovertemplate='Adam<br>步数: %{x}<br>损失: %{y:.4f}<extra></extra>'
))

fig2.update_layout(
    title=dict(text='收敛曲线', font=dict(size=18, color='#2c3e50')),
    width=CONFIG["fig_width"], height=500, margin=dict(l=10, r=10, t=70, b=10),
    xaxis=dict(title='迭代步数', range=[0, common_steps]),
    yaxis=dict(title='损失值', type='log', gridcolor='lightgray', zeroline=False),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    hovermode='x unified', template='plotly_white'
)
fig2.show()

# ============================================================
# 结果分析
# ============================================================
print("=" * 70)
print("【高度病态凸结果分析】")
print("=" * 70)
print(f"RMSProp 最终位置: ({path_rms[-1, 0]:.4f}, {path_rms[-1, 1]:.4f}), f = {path_rms[-1, 2]:.6f}")
print(f"Adam 最终位置: ({path_adam[-1, 0]:.4f}, {path_adam[-1, 1]:.4f}), f = {path_adam[-1, 2]:.6f}")
print("=" * 70)

1. 图像解读：

>1.1 Adam（橙色虚线）—— 势如破竹（带有微瑕）
- **总体趋势**：Adam 的损失值在 2000 步内呈**包络线指数级下降**。从宏观上看，它一路顺畅地冲向了最优点，最终降至极低的 \(10^{-88}\) 级别。
- **微观细节**：如果放大局部（如 350~600 步），你会发现 Adam 的下降曲线**并非绝对平滑**，而是存在极为密集的高频“毛刺”（震荡）。这说明它在高速下降的同时，在陡峭的 Y 轴方向上经历了多次“越界—拉回”的微调。这些震荡是**衰减振荡**，振幅随着迭代次数增加而急剧缩小。
- **为何能成功**：由于加入了冲量，Adam 能在平缓的 X 轴方向不断积累速度，而在陡峭的 Y 轴方向，这些微小的震荡被极大地阻尼，只会留下极其微小的痕迹，丝毫不影响其宏观上“笔直冲向最优点”的壮举。

---

2. 为什么 RMSProp 会“走过头”又“返回”（Zigzag）？

- **自适应步长**：RMSProp 通过除以梯度的均方根（\(\sqrt{s}\)）来归一化每个维度的步长。在 Y 轴上，有效步长被压缩到约等于 `lr * sign(gy)`，即 0.1。
- **缺乏刹车**：当它从 \(y=2.0\) 下滑到接近 \(y=0.05\) 时，它依然会减去 0.1，导致越过最优点（变为 \(y=-0.05\)）。
- **原路弹回**：越过 0 点后，梯度方向立即反转，由于 RMSProp **没有动量记忆**，它只能根据当前的梯度原路跳回 \(y=0.05\)，再跳回 \(-0.05\)，形成高频震荡。
- **最终结果**：它永远卡在 \([-0.05, 0.05]\) 这个区间，损失值 \(f = 50y^2\) 无法下降。在收敛曲线上表现为“平躺”，在等高线图上表现为极小区域内密集的锯齿点。

---

3. 为什么 Adam 也有“小范围走过头”，但能成功？

Adam 并**没有完全消除震荡**，而是将其极大地阻尼，变成幅度递减的“衰减振荡”。放大 350~600 步的收敛曲线，可以看到 Adam 也有尖锐的“毛刺”。

- **惯性导致越界**：Adam 的动量 \(m\) 带来了惯性，使其冲过 0 点。
- **刹车机制**：越过 0 后梯度反转，导致 \(m\) 开始反转，同时二阶矩 \(v\) 增大，形成强力刹车。
- **核心区别**：
  - **RMSProp** 的震荡是**固定步长（0.1）的原地死循环**，导致卡死。
  - **Adam** 的震荡是**依赖惯性的高频微调**。随着 \(t\) 增加，动量 \(m\) 的正负值相互抵消，步长急剧缩小，最终像弹簧一样吸附在 0 上，进而顺着 X 轴继续加速下降。

---

4. 深入探讨：为何冲量（动量）反而有利于减少震荡？

从直觉上看，惯性似乎会增加震荡，但实际是一剂“减震器”，原因有三：

4.1 物理直觉：高频阻尼（减震弹簧）
- **无冲量（RMSProp）**：像坐在无减震器的硬板车上，路面（陡峭 Y 轴）稍有颠簸，头部就会同步且过度地上下猛砸，缺乏缓冲。
- **有冲量（Adam）**：像加了液压减震器。车身因惯性不会瞬间跟着每个小坑起伏，而是保持平稳趋势，平滑掉了高频震荡。

4.2 数学原理：低通滤波器
动量本质是对梯度的**指数加权平均**（\(m_t = \beta m_{t-1} + (1-\beta)g_t\)）。
- **梯度高频交替时**（如 +100, -100, +100...）：无冲量时完全跟随瞬间梯度；有冲量时，正负梯度相互抵消（例如 \(0.9 \times 100 + 0.1 \times (-100) = 80\)），让运动方向保持惯性，只有当反向梯度足够大时才被慢慢纠正。

4.3 几何视角：把锯齿变成平滑螺旋
- **沿谷底 X 轴**：梯度方向始终一致，动量 \(m_x\) 不断累积，加速下山。
- **横跨山谷 Y 轴**：梯度频繁反转。冲量 \(m_y\) 像一场拉锯战，在跨过 0 后，反向梯度刚开始作用，由于惯性，步长急剧缩小，形成幅度越来越小的衰减震荡，最终平稳停在谷底。

---

5. 一句话总结
> **RMSProp 像只看脚下的盲人，路面一颠簸就猛烈调整，导致原地疯狂打转；而 Adam 像看着远方的跑步者，把路面的颠簸当作噪音平滑掉，用积蓄的惯性稳稳地沿着山谷滑行。冲量通过平均历史梯度，过滤掉了陡峭方向的极高频噪音，从而极大地抑制了震荡。**

---

### 算例二：六军对垒——六种优化器在高度病态凸上的全面比拼

在前面的算例中，我们只对比了 Adam 和 RMSProp。为了更全面地理解 Adam 在优化器家族中的地位，我们将战场扩大到 **六种主流优化器**：

- **Adam**（动量为 0.9，自适应二阶矩）
- **RMSProp**（自适应二阶矩，无动量）
- **Adagrad**（自适应累积平方梯度，无动量）
- **Rprop**（仅依赖梯度符号，无动量）
- **HB（Heavy-Ball 动量法）**（纯动量，无自适应）
- **NAG（Nesterov 加速梯度）**（前瞻性动量，无自适应）

这六种优化器涵盖了“纯动量”、“纯自适应”、“动量+自适应”、“纯符号”等多个流派。在高度病态凸函数 $f(x,y)=\tfrac12(x^2 + 100 y^2$) 上，它们将展现出截然不同的命运——有的势如破竹，有的陷入死循环，有的提前衰竭。

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 【全局超参配置区】
# ============================================================
CONFIG = {
    "start_x": -2.0,           
    "start_y": 2.0,            
    "steps": 2000,              
    
    "lr": 0.1,                 
    "beta1": 0.9,
    "beta2": 0.999,
    
    "x_min": -4.0,
    "x_max": 4.0,
    "y_min": -4.0,
    "y_max": 4.0,
    "mesh_res": 400,
    
    "fig_height": 700,
    "fig_width": 1000,
    "path_line_width": 1.2,    
    "path_marker_size": 2.0,   
    "title_x": 0.02,
}

LAMBDA_X = 1.0   
LAMBDA_Y = 100.0     

def f(x, y):
    return 0.5 * (LAMBDA_X * x**2 + LAMBDA_Y * y**2)

def grad_f(x, y):
    dx = LAMBDA_X * x
    dy = LAMBDA_Y * y
    return dx, dy

# ============================================================
# 优化器实现
# ============================================================
def rmsprop_optimizer(start_x, start_y, steps):
    lr = CONFIG["lr"]
    x, y = start_x, start_y
    sx, sy = 0.0, 0.0
    path = [(x, y, f(x, y))]
    for _ in range(steps):
        gx, gy = grad_f(x, y)
        sx = 0.9 * sx + (1 - 0.9) * gx**2
        sy = 0.9 * sy + (1 - 0.9) * gy**2
        x = x - lr * gx / (np.sqrt(sx) + 1e-8)
        y = y - lr * gy / (np.sqrt(sy) + 1e-8)
        path.append((x, y, f(x, y)))
    return np.array(path)

def adam_optimizer(start_x, start_y, steps):
    lr = CONFIG["lr"]
    x, y = start_x, start_y
    mx, my = 0.0, 0.0
    vx, vy = 0.0, 0.0
    t = 0
    path = [(x, y, f(x, y))]
    for _ in range(steps):
        t += 1
        gx, gy = grad_f(x, y)
        mx = 0.9 * mx + (1 - 0.9) * gx
        my = 0.9 * my + (1 - 0.9) * gy
        vx = 0.999 * vx + (1 - 0.999) * gx**2
        vy = 0.999 * vy + (1 - 0.999) * gy**2
        m_hat_x = mx / (1 - 0.9**t)
        m_hat_y = my / (1 - 0.9**t)
        v_hat_x = vx / (1 - 0.999**t)
        v_hat_y = vy / (1 - 0.999**t)
        x = x - lr * m_hat_x / (np.sqrt(v_hat_x) + 1e-8)
        y = y - lr * m_hat_y / (np.sqrt(v_hat_y) + 1e-8)
        path.append((x, y, f(x, y)))
    return np.array(path)

def adagrad_optimizer(start_x, start_y, steps):
    lr = CONFIG["lr"]
    x, y = start_x, start_y
    gx_sq_sum, gy_sq_sum = 0.0, 0.0
    path = [(x, y, f(x, y))]
    for _ in range(steps):
        gx, gy = grad_f(x, y)
        gx_sq_sum += gx**2
        gy_sq_sum += gy**2
        x = x - lr * gx / (np.sqrt(gx_sq_sum) + 1e-8)
        y = y - lr * gy / (np.sqrt(gy_sq_sum) + 1e-8)
        path.append((x, y, f(x, y)))
    return np.array(path)

def rprop_optimizer(start_x, start_y, steps):
    eta_plus = 1.2
    eta_minus = 0.5
    step_min = 1e-6
    step_max = 50.0
    step_x = 0.1
    step_y = 0.1
    
    x, y = start_x, start_y
    prev_gx, prev_gy = 0.0, 0.0
    path = [(x, y, f(x, y))]
    
    for _ in range(steps):
        gx, gy = grad_f(x, y)
        if prev_gx * gx > 0:
            step_x = min(step_x * eta_plus, step_max)
        elif prev_gx * gx < 0:
            step_x = max(step_x * eta_minus, step_min)
        if prev_gy * gy > 0:
            step_y = min(step_y * eta_plus, step_max)
        elif prev_gy * gy < 0:
            step_y = max(step_y * eta_minus, step_min)
        x = x - np.sign(gx) * step_x
        y = y - np.sign(gy) * step_y
        prev_gx, prev_gy = gx, gy
        path.append((x, y, f(x, y)))
    return np.array(path)

def hb_optimizer(start_x, start_y, steps):
    lr = CONFIG["lr"] * 0.05 
    mu = 0.9
    x, y = start_x, start_y
    vx, vy = 0.0, 0.0
    path = [(x, y, f(x, y))]
    for _ in range(steps):
        gx, gy = grad_f(x, y)
        vx = mu * vx - lr * gx
        vy = mu * vy - lr * gy
        x = x + vx
        y = y + vy
        path.append((x, y, f(x, y)))
    return np.array(path)

def nag_optimizer(start_x, start_y, steps):
    lr = CONFIG["lr"] * 0.05
    mu = 0.9
    x, y = start_x, start_y
    vx, vy = 0.0, 0.0
    path = [(x, y, f(x, y))]
    for _ in range(steps):
        x_lookahead = x + mu * vx
        y_lookahead = y + mu * vy
        gx, gy = grad_f(x_lookahead, y_lookahead)
        vx = mu * vx - lr * gx
        vy = mu * vy - lr * gy
        x = x + vx
        y = y + vy
        path.append((x, y, f(x, y)))
    return np.array(path)


start_x = CONFIG["start_x"]
start_y = CONFIG["start_y"]
common_steps = CONFIG["steps"]

print("=" * 70)
print("【高度病态凸理论实验】")
print("=" * 70)
print(f"函数特征值: λ_x = {LAMBDA_X}, λ_y = {LAMBDA_Y} (极度病态)")
print(f"绝对公平学习率: α = {CONFIG['lr']}")
print(f"迭代步数: {common_steps}")
print("注意：HB和NAG为纯动量法，为防溢出已单独调小学习率")
print("=" * 70)

path_rms = rmsprop_optimizer(start_x, start_y, common_steps)
path_adam = adam_optimizer(start_x, start_y, common_steps)
path_adagrad = adagrad_optimizer(start_x, start_y, common_steps)
path_rprop = rprop_optimizer(start_x, start_y, common_steps)
path_hb = hb_optimizer(start_x, start_y, common_steps)
path_nag = nag_optimizer(start_x, start_y, common_steps)

# ============================================================
# 生成等高线数据
# ============================================================
xs = np.linspace(CONFIG["x_min"], CONFIG["x_max"], CONFIG["mesh_res"])
ys = np.linspace(CONFIG["y_min"], CONFIG["y_max"], CONFIG["mesh_res"])
X, Y = np.meshgrid(xs, ys)
Z = f(X, Y)

# ============================================================
# 绘制图1：等高线路径对比
# ============================================================
fig1 = go.Figure()

fig1.add_trace(go.Contour(
    x=xs, y=ys, z=Z,
    colorscale="Viridis",
    contours=dict(start=0, end=300, size=10, showlabels=False, coloring='fill'),
    line=dict(width=0.5, color='black', dash='solid'),
    showscale=True, opacity=0.6, name="等高线（高度病态凸）"
))

# 全部采用点线，极高对比度配色
# 1. RMSProp: 纯正大红（最红）
fig1.add_trace(go.Scatter(x=path_rms[:, 0], y=path_rms[:, 1], mode="lines+markers",
                          line=dict(color="#FF0000", width=CONFIG["path_line_width"], dash="dot"),
                          marker=dict(size=CONFIG["path_marker_size"], color="#FF0000", symbol="circle"), 
                          name="RMSProp"))

# 2. Adam: 纯黑
fig1.add_trace(go.Scatter(x=path_adam[:, 0], y=path_adam[:, 1], mode="lines+markers",
                          line=dict(color="#000000", width=CONFIG["path_line_width"], dash="dot"),
                          marker=dict(size=CONFIG["path_marker_size"], color="#000000", symbol="square"), 
                          name="Adam"))

# 3. Adagrad: 金黄
fig1.add_trace(go.Scatter(x=path_adagrad[:, 0], y=path_adagrad[:, 1], mode="lines+markers",
                          line=dict(color="#FFD700", width=CONFIG["path_line_width"], dash="dot"),
                          marker=dict(size=CONFIG["path_marker_size"], color="#FFD700", symbol="diamond"), 
                          name="Adagrad"))

# 4. Rprop: 亮翠绿
fig1.add_trace(go.Scatter(x=path_rprop[:, 0], y=path_rprop[:, 1], mode="lines+markers",
                          line=dict(color="#00FF00", width=CONFIG["path_line_width"], dash="dot"),
                          marker=dict(size=CONFIG["path_marker_size"], color="#00FF00", symbol="triangle-up"), 
                          name="Rprop"))

# 5. HB: 纯蓝
fig1.add_trace(go.Scatter(x=path_hb[:, 0], y=path_hb[:, 1], mode="lines+markers",
                          line=dict(color="#0000FF", width=CONFIG["path_line_width"], dash="dot"),
                          marker=dict(size=CONFIG["path_marker_size"], color="#0000FF", symbol="cross"), 
                          name="HB"))

# 6. NAG: 深紫
fig1.add_trace(go.Scatter(x=path_nag[:, 0], y=path_nag[:, 1], mode="lines+markers",
                          line=dict(color="#800080", width=CONFIG["path_line_width"], dash="dot"),
                          marker=dict(size=CONFIG["path_marker_size"], color="#800080", symbol="x"), 
                          name="NAG"))

fig1.add_trace(go.Scatter(x=[start_x], y=[start_y], mode="markers", marker=dict(size=10, color="#ffcc00"), name="起点"))
fig1.add_trace(go.Scatter(x=[0], y=[0], mode="markers", marker=dict(size=12, color="#00ff00", symbol="star", line=dict(color="black", width=1)), name="全局最优"))

fig1.update_layout(
    width=CONFIG["fig_width"], height=CONFIG["fig_height"], template="plotly_white",
    title=dict(text="高度病态凸下：六种优化器路径对比", x=CONFIG["title_x"], xanchor="left", font=dict(size=18)),
    xaxis_title="x", yaxis_title="y",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5, font=dict(size=11)),
    hovermode='closest', margin=dict(t=80, r=80)
)
fig1.show()

# ============================================================
# 绘制图2：收敛曲线
# ============================================================
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=list(range(len(path_rms))), y=path_rms[:, 2], mode='lines', name='RMSProp',
    line=dict(color="#FF0000", width=CONFIG["path_line_width"], dash='dot')  # 纯正大红
))

fig2.add_trace(go.Scatter(
    x=list(range(len(path_adam))), y=path_adam[:, 2], mode='lines', name='Adam',
    line=dict(color="#000000", width=CONFIG["path_line_width"], dash='dot')  # 纯黑
))

fig2.add_trace(go.Scatter(
    x=list(range(len(path_adagrad))), y=path_adagrad[:, 2], mode='lines', name='Adagrad',
    line=dict(color="#FFD700", width=CONFIG["path_line_width"], dash='dot')   # 金黄
))

fig2.add_trace(go.Scatter(
    x=list(range(len(path_rprop))), y=path_rprop[:, 2], mode='lines', name='Rprop',
    line=dict(color="#00FF00", width=CONFIG["path_line_width"], dash='dot') # 亮翠绿
))

fig2.add_trace(go.Scatter(
    x=list(range(len(path_hb))), y=path_hb[:, 2], mode='lines', name='HB',
    line=dict(color="#0000FF", width=CONFIG["path_line_width"], dash='dot') # 纯蓝
))

fig2.add_trace(go.Scatter(
    x=list(range(len(path_nag))), y=path_nag[:, 2], mode='lines', name='NAG',
    line=dict(color="#800080", width=CONFIG["path_line_width"], dash='dot') # 深紫
))

fig2.update_layout(
    title=dict(text='收敛曲线对比：六种优化器全面比拼', font=dict(size=18, color='#2c3e50')),
    width=CONFIG["fig_width"], height=500, margin=dict(l=10, r=10, t=70, b=10),
    xaxis=dict(title='迭代步数', range=[0, common_steps]),
    yaxis=dict(title='损失值', type='log', gridcolor='lightgray', zeroline=False),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    hovermode='x unified', template='plotly_white'
)
fig2.show()



这张图完美地展示了六种优化器在高度病态凸函数（$\lambda_x=1, \lambda_y=100$）上的表现。

结合之前的数学原理，我们可以对这张收敛曲线图进行非常深度的拆解：

1. 整体格局：“两大阵营”划分

从纵轴（对数尺度的损失值）来看，六条线分成了截然不同的两个阵营：
- 前排阵营（成功收敛）：Adam（黑虚线）、HB（蓝虚线）、NAG（紫虚线）。它们一路向下，将损失值降到了 $10^{-90}$ 级别以下。
- 后排阵营（陷入停滞）：RMSProp（红虚线）、Rprop（绿点线）、Adagrad（黄点线）。它们在初期下降后，迅速变成水平线，卡死在 $10^0$ 或 $10^{-2}$ 附近。

2. 前排阵营的详细解读（黑、蓝、紫）

- Adam（黑虚线）：表现最稳健。它在初期有一个快速的下降，然后保持极平稳的指数级下降。因为它的自适应二阶矩能够自动归一化 Y 轴的巨大梯度，所以它几乎没有受到病态条件数的影响，一路“势如破竹”。
- HB（蓝虚线）与 NAG（紫虚线）：这两条线呈现波浪形（锯齿状）的下降。这印证了我们之前的理论：动量法（HB和NAG）在陡峭的 Y 轴方向上会反复“越界—拉回”。由于动量缺乏自适应缩放，它们只能靠惯性硬冲，所以每过一段时间就会在 Y 轴上甩出一个深坑（损失值短暂上升），然后再拉回来继续下降。
- HB vs NAG 的区别：仔细观察可以发现，紫色的 NAG 在某些阶段（比如 1500 步之后）比蓝色的 HB 下降得更快。这是因为 NAG 具有“前瞻性”，它能在动量冲过头之前提前感知到陡峭方向的梯度反转，从而更早地踩刹车，因此它的振幅更小，能更有效地将动量转化为 X 轴方向的前进速度。

3. 后排阵营的详细解读（红、绿、黄）

这三条线代表了三种不同的“失败模式”：

- RMSProp（红色点线）：“Zigzag 死循环”。它在第 50 步左右经历了一次剧烈的暴跌，随后瞬间反弹并彻底平躺。正如我们之前分析的，它在 Y 轴上的步长刚好能“跨过”最优点，导致它在原点附近横跳，损失值永远卡在 1 左右。
- Rprop（绿色点线）：“步长缩小导致的瘫痪”。Rprop 只依赖梯度的符号。在病态 Y 轴上，它的步长很快就因为频繁的符号反转而被除以 2、除以 4... 变得极小。它比 RMSProp 多走了一小步（降到 $10^{-2}$ 左右），但随后由于步长过小，彻底失去前进能力。
- Adagrad（黄色点线）：“学习率过早衰竭”。Adagrad 累积了所有历史梯度的平方，导致分母在初始阶段快速膨胀。所以它头几十步走得还行，但后面每一步的步长都微小到可以忽略不计，只能在原点附近肉眼可见地“爬行”。

4. 为什么后排阵营看起来很“粗/密集”？

注意图右侧（1500~2000步）的黑、蓝、紫三条线，上面布满了极密的锯齿。这并非数据错误，而是它们在进行极其微小但高频的震荡。因为对数坐标被极度放大，它们到达底层时，虽然损失值已经很低，但在 Y 轴上依然有微弱的动量反复，体现在图上就是密集的“毛刺”。

5. 最终结论

这张图极其直观地证明了：在极端病态的条件下，带有冲量（Momentum）的优化器（HB、NAG、Adam）是唯一能穿透病态山谷的工具。而纯自适应的 RMSProp / Adagrad，以及纯符号的 Rprop，因为缺乏动量（或动量被提前耗竭），都无法克服 Y 轴上的巨大梯度落差。Adam 因为结合了动量和自适应缩放，成为了这里当之无愧的“最佳王者”。

## 2. 算法原理与核心工作流程

### 2.1 融合两种算法的优点

Adam 的核心设计思想是将 **动量法（Momentum）** 和 **自适应学习率（RMSProp）** 的优势完美融合，同时引入偏差校正机制，使训练初期更加稳定。

#### 带着“惯性”前进（动量思想）
- **核心**：就像一个小球在下山时，不仅看当前最陡的方向，还会 **累积之前的速度和方向**。
- **效果**：面对小坑洼或噪声时能保持稳定，在平坦区域也能加速前进，不容易卡住。
- **实现方式**：通过计算 **梯度的一阶矩估计**（即过去梯度的指数移动平均值）来实现。

#### 为每个参数“量身定制”步长（自适应学习率思想）
- **核心**：不再为所有参数使用统一的学习率，而是为每个参数 **单独调整** 步长。
- **效果**：更新频繁或梯度较大的参数，步长会小一些；更新稀疏或梯度平缓的参数，步长会大一些。
- **实现方式**：通过计算 **梯度的二阶矩估计**（即过去梯度平方的指数移动平均值）来实现。

### 2.2 Adam 的核心工作流程

Adam 的运作可以简化为以下 **5 个步骤**：

| 步骤 | 操作 | 公式 | 说明 |
| :--- | :--- | :--- | :--- |
| 1 | **计算当前梯度** | $g_t = \nabla_\theta f(\theta_{t-1})$ | 和标准梯度下降一样，算出当前位置的梯度方向 |
| 2 | **更新一阶矩估计（$m_t$）** | $m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$ | 类似于动量，记录梯度的“平均方向”。$\beta_1$ 常设为 0.9 |
| 3 | **更新二阶矩估计（$v_t$）** | $v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$ | 类似于 RMSProp，记录梯度平方的“平均大小”。$\beta_2$ 常设为 0.999 |
| 4 | **偏差校正** | $\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$ | 由于 $m_t$ 和 $v_t$ 初始为 0，在训练初期会偏向 0。Adam 会进行修正，让估计更准确 |
| 5 | **更新参数** | $\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$ | 结合修正后的 $\hat{m}_t$（方向）和 $\hat{v}_t$（步长调整），最终更新模型参数 |

### 2.3 关键参数说明

| 参数 | 默认值 | 作用 |
| :--- | :--- | :--- |
| $\eta$ (学习率) | 0.001 | 控制整体更新步长的大小 |
| $\beta_1$ | 0.9 | 一阶矩的衰减率，控制动量的记忆长度 |
| $\beta_2$ | 0.999 | 二阶矩的衰减率，控制自适应步长的记忆长度 |
| $\epsilon$ | $10^{-8}$ | 防止分母为 0 的微小常数 |

### 2.4 偏差校正的必要性

由于 $m_t$ 和 $v_t$ 都从 0 开始初始化，在训练初期（$t$ 较小时），$m_t$ 和 $v_t$ 会严重偏向 0，导致更新步长被过度放大。偏差校正通过除以 $(1 - \beta^t)$ 来补偿这种偏差，使得初期估计更加准确，避免了训练早期的不稳定震荡。

## 3. Adam 的主要优势

- **实现简单，计算高效**：只需要计算一阶梯度，内存需求也不大。
- **超参数“不敏感”**：默认的超参数（$\beta_1=0.9, \beta_2=0.999$）在大多数问题上都表现良好，通常不需要大量调参就能获得不错的结果。
- **适用性强**：无论是处理 **大规模数据集**、**高维参数空间**，还是 **稀疏梯度**、**非平稳目标** 问题，Adam 都表现出色。因此，它在自然语言处理（NLP）和计算机视觉（CV）等前沿领域应用非常广泛。

## 4. 补充小贴士

- **常用变体**：实际应用中，一个流行的变体是 **AdamW**。它在原始 Adam 基础上修正了权重衰减（L2 正则化）的实现方式，在很多任务上表现更好。
- **潜在不足**：虽然 Adam 很强大，但理论上在某些情况下，它的自适应学习率可能导致 **无法收敛到最优解**。因此，在追求极致泛化性能时，有时会在训练后期切换回 SGD。

## 5. 快速总结：Adam vs RMSProp

| 对比维度 | Adam | RMSProp（参考） |
| :--- | :--- | :--- |
| 核心机制 | 动量 + 自适应学习率 | 仅自适应学习率 |
| 一阶矩（动量） | ✅ 有 | ❌ 无 |
| 二阶矩（自适应） | ✅ 有 | ✅ 有 |
| 偏差校正 | ✅ 有 | ❌ 无 |
| 适用场景 | 通用，尤其适合大规模/稀疏/噪声数据 | 适合非平稳目标，但对初始学习率敏感 |
| 常见变体 | AdamW | RMSProp（较少变体） |